In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd
import numpy as np
import gc, os, getpass, copy, random, torch
from torch import nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device = {str(device).title()}")

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
df_path = os.path.join(script_dir, "Datasets","Machine Failure.csv")
df = pd.read_csv(df_path)

df = df.dropna()
df["target"] = df["target"].map({"Failure":1,"Normal":0})

x = df.drop(columns="target").values
y = df["target"].values.reshape(-1,1)

x_scaler = StandardScaler()
x = x_scaler.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.45, random_state=72)
x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

user_account = getpass.getuser()

print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")
print(f"X Test = {x_test.shape}")
print(f"Y Test = {y_test.shape}")

class BinaryClassifier(nn.Module):
    def __init__(self, input_dim=x_train.shape[1]):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=13),
            nn.LeakyReLU(),
            nn.BatchNorm1d(13),

            nn.Linear(in_features=13, out_features=28),
            nn.LeakyReLU(),
            nn.BatchNorm1d(28),

            nn.Linear(in_features=29, out_features=41),
            nn.LeakyReLU(),
            nn.BatchNorm1d(41),

            nn.Linear(in_features=41, out_features=66),
            nn.LeakyReLU(),
            nn.BatchNorm1d(66),
            nn.Dropout(p=0.6),

            nn.Linear(in_features=66, out_features=85),
            nn.LeakyReLU(),
            nn.BatchNorm1d(85),
            nn.Dropout(p=0.9),

            nn.Linear(in_features=85, out_features=109),
            nn.LeakyReLU(0.45),
            nn.BatchNorm1d(109),
            nn.Dropout(p=0.5),

            nn.Linear(in_features=109, out_features=32),
            nn.LeakyReLU(),
            nn.BatchNorm1d(32),

            nn.Linear(in_features=32, out_features=1)
        )

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return self.network(x)
    
losses = []
accuracies = []
def run_grid_search(rates_list, epoch_count_list, seed_val):
    best_state = None
    loss_fn = nn.BCEWithLogitsLoss()

    print(f"== Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<15} | {'Val Loss':<15} | {'Val Accuracy':<15}")
    print("="*65)

    for lr in rates_list:
        for epoch_count in epoch_count_list:
            torch.manual_seed(int(seed_val))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed_val))

            model = BinaryClassifier().to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)

            for epoch in tqdm(range(epoch_count)):
                model.train()
                y_preds = model(x_train)
                loss = loss_fn(y_preds, y_train)    

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.inference_mode():
                test_logits = model(x_test)
                test_loss = loss_fn(test_logits, y_test).item()
                metric = BinaryAccuracy().to(device)
                prec_metric = BinaryPrecision().to(device)
                rec_metric = BinaryRecall().to(device)
                test_acc = metric(torch.sigmoid(test_logits),y_test).item()
                losses.append(test_loss)
                accuracies.append(test_acc)

            print(f"{lr:<15} | {epoch_count:<15} | {test_loss:<15} | {test_acc * 100}")
            best_state = copy.deepcopy(model.state_dict())

            del moderl, optimizer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n● Best Loss = {min(losses):.2f} | Best Accuracy = {max(accuracies):.2f}")
    print(f"● Worst Loss = {max(losses):.2f} | Worst Accuracy = {min(accuracies):.2f}")     
    print(f"● Preicison Score = {prec_metric(torch.sigmoid(test_logits) ,y_test).item()}")
    print(f"● Recall Score = {rec_metric(torch.sigmoid(test_logits), y_test).item()}")                

    return best_state

epoch_count_list = [980, 1020, 780, 810]
rates_list = [0.1, 0.001, 0.01, 0.0001]
seed_val = float(input("-- Enter a Value for a Seed : ").strip())
best_state = run_grid_search(epoch_count_list=epoch_count_list, rates_list=rates_list, seed_val=seed_val)
torch.save(best_state, f"Machine Failure Predictor Parameters.pth")

fresh_model = BinaryClassifier().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

session_inputs = []
session_predictions = []

conversion = {
    1:"Failure",
    0:"Normal"
}

while True:
    user_name = input("-- Enter your Name ( or type 'exit' to quit ) : ").strip()
    if user_name.lower() == "exit" or not user_name:
        print(f"Exiting Interactive Loop ...")
        break

    operating_hours = float(input("-- Enter the Number of Operating Hours : ").strip())
    temperature_c = float(input("-- Enter the Temperature in Celsius : ").strip())
    vibration_level = float(input("-- Enter the Vibration Level : ").strip())
    load_pct = float(input("-- Enter the Load Percentage : ").strip())
    maintenance_gap_days = float(input("-- Enter the Maintenance Gap in Days: ").strip())

    raw_array = np.array([operating_hours, temperature_c, vibration_level, load_pct, maintenance_gap_days])
    scaled_tensor = torch.tensor(x_scaler.transform(raw_array), dtype=torch.float32).to(device)

    with torch.inference_mode():
        logits = fresh_model(scaled_tensor)
        probability = torch.sigmoid(logits).item()

    print(f"Predicted Machine Failure Status = {conversion.get(probability, 0)}")
    session_inputs.append(f"{user_name.title()} ( {temperature_c} ° C )")
    session_predictions.append(probability * 100)

try:
    onedrive = input("-- Do you use One Drive or Not : ").strip().title()
    if onedrive == "Yes":
        desktop_path = f"C:/Users/{user_account}/OneDrive/Desktop/Machine Failure Predictor Details.txt"
    elif onedrive == "No":
        desktop_path = f"C:/Users/{user_account}/Desktop/Machine Failure Predictor Details.txt"
except BaseException as e:
    print(f"Invalid Input. Error has occured = {e}")

try:
    with open(desktop_path, "w", encoding="utf-8") as file:
        file.write("== Machine Failure Predictor Details")
        file.write(f"\n● Parameter Weights = {fresh_model.network[0].weight[0][0].item():.3f}\n")     
        file.write(f"● Parameter Bias = {fresh_model.network[0].bias[0].item():.3f}\n")     
        file.write(f"● Best Loss = {min(losses):.3f} | Best Accuracy = {max(accuracies):.3f}\n")
        file.write(f"● Worst Loss = {max(losses):.3f} | Worst Accurcy = {min(accuracies):.3f}\n")
    print(f"\nSuccessfully Saved the Network Details")
except BaseException as e:
    print(f"Failed to Save Network Details = {e}")    
